# LeetCode 2511 — Maximum Enemy Forts That Can Be Captured bằng Python

Link đề: https://leetcode.com/problems/maximum-enemy-forts-that-can-be-captured/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def captureForts(self, forts: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cách đọc mảng `forts` gồm ba giá trị `-1` (vị trí trống), `0` (pháo đài địch), `1` (pháo đài của ta) và điều kiện di chuyển: chỉ đi **qua pháo đài địch (các số `0`)**.
2. Nhận ra bài toán là tìm **đoạn các số `0` liên tiếp** nằm **giữa một số `1` và một số `-1`** (theo cả hai chiều) rồi lấy đoạn dài nhất.
3. Biết quét mảng một lần với **chỉ số phần tử khác 0 gần nhất** (`last`), thay vì thử mọi cặp.
4. Viết 4 cách: thử mọi cặp `(i, j)`, mở rộng từ mỗi số `1` sang hai phía, quét một lần với `last`, và biểu thức chính quy (`re`) với lookahead; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: đếm hết mọi số `0` của mảng, chỉ xét một chiều, không kiểm tra điểm kết thúc là `-1`, và nối hai phần tử "cùng loại" (`1` với `1` hoặc `-1` với `-1`).

## 2. Đề bài & ví dụ

Cho mảng `forts` độ dài `n`, trong đó:

- `forts[i] = 1`: vị trí `i` là pháo đài **của bạn**;
- `forts[i] = 0`: vị trí `i` là pháo đài của **kẻ địch**;
- `forts[i] = -1`: vị trí `i` **trống**.

Bạn muốn đưa quân từ một pháo đài của bạn tại `i` đến một vị trí **trống** `j` sao cho **mọi vị trí nằm giữa** `i` và `j` đều là pháo đài địch (`forts[k] == 0` với `min(i, j) < k < max(i, j)`). Khi đó toàn bộ pháo đài địch trên đường đi bị chiếm.

Trả về **số pháo đài địch tối đa** có thể chiếm bằng một lần di chuyển. Nếu không thể di chuyển (không có pháo đài của bạn hoặc không có vị trí trống phù hợp) thì trả về `0`.

| Ví dụ | forts | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,0,0,-1,0,0,0,0,1]` | `4` | từ vị trí `8` sang vị trí `3` qua bốn số `0`; từ vị trí `0` sang `3` chỉ chiếm 2 |
| 2 | `[0,0,1,-1]` | `0` | `1` và `-1` kề nhau nên không chiếm được pháo đài nào |

Constraints: `1 <= forts.length <= 1000`; `-1 <= forts[i] <= 1`.

In [1]:
examples = [([1, 0, 0, -1, 0, 0, 0, 0, 1], 4), ([0, 0, 1, -1], 0)]
for forts, expected in examples:
    best = 0
    n = len(forts)
    for i in range(n):
        for j in range(n):
            if forts[i] == 1 and forts[j] == -1:
                lo, hi = min(i, j), max(i, j)
                if all(forts[k] == 0 for k in range(lo + 1, hi)):
                    print(f"  forts={forts}: đi {i} -> {j} chiếm {hi - lo - 1}")
                    best = max(best, hi - lo - 1)
    print("kết quả:", best, "(mong đợi", expected, ")")

  forts=[1, 0, 0, -1, 0, 0, 0, 0, 1]: đi 0 -> 3 chiếm 2
  forts=[1, 0, 0, -1, 0, 0, 0, 0, 1]: đi 8 -> 3 chiếm 4
kết quả: 4 (mong đợi 4 )
  forts=[0, 0, 1, -1]: đi 2 -> 3 chiếm 0
kết quả: 0 (mong đợi 0 )


## 3. Kiến thức nền

### 3.1. Chỉ số và đoạn giữa hai chỉ số

Với hai chỉ số `i < j`, các vị trí **nằm giữa** là `i + 1, ..., j - 1`, tức `range(i + 1, j)`; có `j - i - 1` vị trí. Nếu hai vị trí kề nhau (`j = i + 1`) thì không có vị trí nào ở giữa.

In [2]:
i, j = 3, 8
print(list(range(i + 1, j)), "-> số vị trí giữa:", j - i - 1)
print(list(range(4, 5)), "-> kề nhau:", 5 - 4 - 1)

[4, 5, 6, 7] -> số vị trí giữa: 4
[4] -> kề nhau: 0


### 3.2. `all()` và `any()` với generator

`all(cond for k in ...)` kiểm tra **mọi** phần tử thỏa; trả `True` với dãy rỗng (đoạn giữa rỗng thì "mọi vị trí giữa đều là 0" là đúng). `any(...)` kiểm tra có **ít nhất một**.

In [3]:
forts = [1, 0, 0, -1]
print(all(forts[k] == 0 for k in range(1, 3)))      # True: các vị trí 1, 2 đều là 0
print(all(forts[k] == 0 for k in range(1, 1)))      # True: dãy rỗng
print(any(forts[k] == -1 for k in range(0, 4)))     # True

True
True
True


### 3.3. Duyệt với `enumerate` và nhớ "phần tử khác 0 gần nhất"

Khi quét mảng từ trái sang phải, ta chỉ cần nhớ **chỉ số `last` của phần tử khác `0` gần nhất** ở bên trái. Mỗi khi gặp phần tử khác `0` mới tại `i`, đoạn giữa `last` và `i` toàn là `0` (vì không có phần tử khác 0 nào giữa chúng).

In [4]:
forts = [1, 0, 0, -1, 0, 1]
last = -1
for i, v in enumerate(forts):
    if v != 0:
        print(f"gặp {v} tại {i}; phần tử khác 0 trước đó ở chỉ số {last}")
        last = i

gặp 1 tại 0; phần tử khác 0 trước đó ở chỉ số -1
gặp -1 tại 3; phần tử khác 0 trước đó ở chỉ số 0
gặp 1 tại 5; phần tử khác 0 trước đó ở chỉ số 3


### 3.4. Biểu thức chính quy `re` và lookahead

`re` tìm mẫu trong chuỗi. `0*` nghĩa là "không hoặc nhiều ký tự 0". Lookahead `(?=(mẫu))` tìm mọi vị trí **kể cả chồng lấn nhau** và trả về phần khớp trong nhóm; nếu không dùng lookahead thì `findall` bỏ qua các khớp chồng lấn.

In [5]:
import re
s = "A00B00A"                                         # A = 1, B = -1, 0 = 0
print(re.findall(r"A0*B", s), re.findall(r"B0*A", s))
s2 = "B0A0B"
print(re.findall(r"B0*A|A0*B", s2))                  # bỏ lỡ 'A0B' vì chồng lấn
print(re.findall(r"(?=(B0*A|A0*B))", s2))            # lookahead: tìm đủ cả hai

['A00B'] ['B00A']
['B0A']
['B0A', 'A0B']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `[1,0,0,-1,0,0,0,0,1]` (chỉ số 0..8):

- Phần tử khác `0` ở các chỉ số `0 (1)`, `3 (-1)`, `8 (1)`.
- Giữa chỉ số `0` và `3`: hai số `0`, nhưng `1` và `-1` **khác nhau** nên là một cặp hợp lệ: chiếm `3 - 0 - 1 = 2`.
- Giữa chỉ số `3` và `8`: bốn số `0`, `-1` và `1` khác nhau nên cũng hợp lệ: chiếm `8 - 3 - 1 = 4`.
- Kết quả `max(2, 4) = 4`.

Hai phần tử khác `0` **liên tiếp** (không có số khác `0` ở giữa) tạo thành một "cặp hợp lệ" **khi và chỉ khi một là `1` và một là `-1`** (khác dấu); chiều đi (từ `1` đến `-1` hay ngược lại) không quan trọng vì đề cho phép cả hai hướng. Giá trị của cặp là số số `0` ở giữa.

In [6]:
forts = [1, 0, 0, -1, 0, 0, 0, 0, 1]
nz = [(i, v) for i, v in enumerate(forts) if v != 0]
print("các phần tử khác 0:", nz)
for (i1, v1), (i2, v2) in zip(nz, nz[1:]):
    print(f"cặp ({i1}:{v1}) - ({i2}:{v2}) -> khác dấu? {v1 != v2}, số 0 ở giữa = {i2 - i1 - 1}")

các phần tử khác 0: [(0, 1), (3, -1), (8, 1)]
cặp (0:1) - (3:-1) -> khác dấu? True, số 0 ở giữa = 2
cặp (3:-1) - (8:1) -> khác dấu? True, số 0 ở giữa = 4


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp `(i, j)`

**Ý tưởng:** với mọi `i` mà `forts[i] == 1` và mọi `j` mà `forts[j] == -1`, kiểm tra mọi vị trí giữa đều là `0`; nếu thỏa, ứng viên là `|i - j| - 1`. Lấy max.

In [7]:
from typing import List

class SolutionBrute:
    def captureForts(self, forts: List[int]) -> int:
        n = len(forts)
        best = 0
        for i in range(n):
            if forts[i] != 1:
                continue
            for j in range(n):
                if forts[j] != -1:
                    continue
                lo, hi = min(i, j), max(i, j)
                if all(forts[k] == 0 for k in range(lo + 1, hi)):
                    best = max(best, hi - lo - 1)
        return best

In [8]:
sol = SolutionBrute()
print(sol.captureForts([1, 0, 0, -1, 0, 0, 0, 0, 1]))   # 4
print(sol.captureForts([0, 0, 1, -1]))                  # 0

4
0


**Độ phức tạp:** Time `O(n^3)` ở trường hợp xấu (hai vòng lặp cặp và một lần kiểm tra đoạn giữa); với `n <= 1000` vẫn là nặng. Space `O(1)`. **Nút thắt:** kiểm tra lại đoạn giữa cho từng cặp, trong khi chỉ cần xét cặp phần tử khác 0 **liền nhau**.

## 6. Ý tưởng thuật toán / bước đột phá

Một cặp `(i, j)` hợp lệ cần mọi vị trí giữa là `0`, nghĩa là **không có phần tử khác 0 nào giữa `i` và `j`**: `i` và `j` là **hai phần tử khác 0 liền kề nhau** trong danh sách các phần tử khác 0. Vì vậy chỉ có tối đa `n - 1` cặp ứng viên.

- Quét từ trái sang phải, giữ `last` là chỉ số phần tử khác `0` gần nhất.
- Gặp phần tử khác 0 tại `i` thì nếu `last` tồn tại và `forts[last] != forts[i]` (một `1`, một `-1`), cập nhật `best = max(best, i - last - 1)`.
- Sau đó đặt `last = i`.

**Quy tắc:** *"hai phần tử khác 0 liền nhau mà khác dấu thì đoạn 0 ở giữa là một lần chiếm hợp lệ."*

**Invariant:** trước khi xử lý chỉ số `i`, `last` là chỉ số phần tử khác 0 lớn nhất `< i` (hoặc `-1` nếu chưa có), và mọi số trong `forts[last+1..i-1]` đều là `0`.

## 7. Các cách tiếp theo

### Cách 2 — Mở rộng từ mỗi số `1` sang hai phía

Với mỗi vị trí `i` có `forts[i] == 1`, đi sang **phải** qua các số `0`; nếu dừng lại ở một số `-1` thì cập nhật đáp án bằng số bước. Làm tương tự sang **trái**. Mỗi đoạn `0` bị quét tối đa hai lần (từ hai đầu), nên tổng vẫn tuyến tính.

In [9]:
from typing import List

class SolutionExpand:
    def captureForts(self, forts: List[int]) -> int:
        n = len(forts)
        best = 0
        for i in range(n):
            if forts[i] != 1:
                continue
            for step in (1, -1):                       # phải, rồi trái
                j = i + step
                while 0 <= j < n and forts[j] == 0:
                    j += step
                if 0 <= j < n and forts[j] == -1:
                    best = max(best, abs(j - i) - 1)
        return best

In [10]:
sol = SolutionExpand()
print(sol.captureForts([1, 0, 0, -1, 0, 0, 0, 0, 1]))   # 4
print(sol.captureForts([0, 0, 1, -1]))                  # 0

4
0


**Độ phức tạp:** Time `O(n)` (mỗi số `0` bị quét tối đa từ hai số khác 0 hai đầu), Space `O(1)`.

### Cách 3 — Quét một lần với `last` (cách nộp bài)

Đúng như mục 6.

In [11]:
from typing import List

class SolutionLast:
    def captureForts(self, forts: List[int]) -> int:
        best = 0
        last = -1                                      # chỉ số phần tử khác 0 gần nhất
        for i, v in enumerate(forts):
            if v == 0:
                continue
            if last != -1 and forts[last] != v:        # một 1 và một -1
                best = max(best, i - last - 1)
            last = i
        return best

In [12]:
sol = SolutionLast()
print(sol.captureForts([1, 0, 0, -1, 0, 0, 0, 0, 1]))   # 4
print(sol.captureForts([0, 0, 1, -1]))                  # 0

4
0


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Biểu thức chính quy `re`

Đổi `1 -> 'A'`, `-1 -> 'B'`, `0 -> '0'` thành chuỗi. Mọi lần chiếm hợp lệ là một đoạn `A0*B` hoặc `B0*A`; dùng lookahead để không bỏ sót các khớp chồng lấn (hai đầu cùng dùng chung một phần tử). Độ dài khớp trừ `2` là số pháo đài chiếm được.

In [13]:
import re
from typing import List

class SolutionRegex:
    def captureForts(self, forts: List[int]) -> int:
        s = "".join({1: "A", 0: "0", -1: "B"}[v] for v in forts)
        best = 0
        for m in re.findall(r"(?=(A0*B|B0*A))", s):
            best = max(best, len(m) - 2)
        return best

In [14]:
sol = SolutionRegex()
print(sol.captureForts([1, 0, 0, -1, 0, 0, 0, 0, 1]))   # 4
print(sol.captureForts([0, 0, 1, -1]))                  # 0

4
0


**Độ phức tạp:** Time `O(n)` cho mỗi mẫu trong trường hợp thường gặp (có thể xấu hơn với mẫu tổng quát, nhưng ở đây `0*` rất đơn giản), Space `O(n)` cho chuỗi. Cách này hay để biết, nhưng cách 3 rõ ràng hơn.

## 8. TRACE từng bước

Trace Cách 3 với `forts = [1,0,0,-1,0,0,0,0,1]`:

| i | `v` | `last` trước | Điều kiện | `best` | `last` sau |
|---|---|---|---|---|---|
| 0 | 1 | -1 | chưa có `last` | 0 | 0 |
| 1, 2 | 0 | 0 | bỏ qua | 0 | 0 |
| 3 | -1 | 0 | `1 != -1` -> `3 - 0 - 1 = 2` | 2 | 3 |
| 4..7 | 0 | 3 | bỏ qua | 2 | 3 |
| 8 | 1 | 3 | `-1 != 1` -> `8 - 3 - 1 = 4` | 4 | 8 |

Kết quả `4`.

In [15]:
def trace_last(forts):
    best, last = 0, -1
    print("forts =", forts)
    for i, v in enumerate(forts):
        if v == 0:
            print(f"  i={i}: v=0 -> bỏ qua")
            continue
        note = "chưa có last"
        if last != -1:
            if forts[last] != v:
                cand = i - last - 1
                best = max(best, cand)
                note = f"khác dấu với forts[{last}]={forts[last]} -> chiếm {cand}"
            else:
                note = f"cùng loại với forts[{last}]={forts[last]} -> không hợp lệ"
        print(f"  i={i}: v={v:2d}, last={last}: {note}, best={best}")
        last = i
    print("kết quả:", best)
    return best

trace_last([1, 0, 0, -1, 0, 0, 0, 0, 1])
trace_last([1, 0, 1, 0, -1])
trace_last([0, 0, 1, -1])

forts = [1, 0, 0, -1, 0, 0, 0, 0, 1]
  i=0: v= 1, last=-1: chưa có last, best=0
  i=1: v=0 -> bỏ qua
  i=2: v=0 -> bỏ qua
  i=3: v=-1, last=0: khác dấu với forts[0]=1 -> chiếm 2, best=2
  i=4: v=0 -> bỏ qua
  i=5: v=0 -> bỏ qua
  i=6: v=0 -> bỏ qua
  i=7: v=0 -> bỏ qua
  i=8: v= 1, last=3: khác dấu với forts[3]=-1 -> chiếm 4, best=4
kết quả: 4
forts = [1, 0, 1, 0, -1]
  i=0: v= 1, last=-1: chưa có last, best=0
  i=1: v=0 -> bỏ qua
  i=2: v= 1, last=0: cùng loại với forts[0]=1 -> không hợp lệ, best=0
  i=3: v=0 -> bỏ qua
  i=4: v=-1, last=2: khác dấu với forts[2]=1 -> chiếm 1, best=1
kết quả: 1
forts = [0, 0, 1, -1]
  i=0: v=0 -> bỏ qua
  i=1: v=0 -> bỏ qua
  i=2: v= 1, last=-1: chưa có last, best=0
  i=3: v=-1, last=2: khác dấu với forts[2]=1 -> chiếm 0, best=0
kết quả: 0


0

## 9. Code nộp LeetCode

Cách quét một lần với `last`:

In [16]:
from typing import List

class Solution:
    def captureForts(self, forts: List[int]) -> int:
        best = 0
        last = -1
        for i, v in enumerate(forts):
            if v == 0:
                continue
            if last != -1 and forts[last] != v:
                best = max(best, i - last - 1)
            last = i
        return best

In [17]:
print(Solution().captureForts([1, 0, 0, -1, 0, 0, 0, 0, 1]))
print(Solution().captureForts([0, 0, 1, -1]))

4
0


## 10. Vì sao thuật toán đúng

1. **Cặp hợp lệ = hai phần tử khác 0 liền kề.** Đường đi từ `i` đến `j` chỉ qua các số `0`. Nếu giữa `i` và `j` có một phần tử khác 0, đường đi bị chặn. Vậy `i`, `j` phải là hai phần tử khác 0 liền kề nhau.
2. **Điều kiện về loại.** Điểm xuất phát phải là `1` và đích phải là `-1` (hoặc ngược lại về vị trí `j < i`). Hai phần tử liền kề cùng loại (`1,1` hoặc `-1,-1`) không tạo đường đi hợp lệ, hai phần tử khác loại thì có.
3. **Hai chiều.** Đề cho `j` có thể nhỏ hơn hoặc lớn hơn `i`. Cặp liền kề `(a, b)` với `a` ở trái `b`: nếu `a = 1`, `b = -1` đi sang phải; nếu `a = -1`, `b = 1` đi sang trái. Cả hai đều hợp lệ nên điều kiện `forts[last] != v` bao quát.
4. **Giá trị chiếm được.** Số số `0` giữa `last` và `i` đúng bằng `i - last - 1`.
5. **Bao phủ.** Mỗi cặp phần tử khác 0 liền kề được xét đúng một lần khi `i` là phần tử sau; vì `last` luôn là phần tử khác 0 liền trước.
6. **Không có cặp hợp lệ.** Nếu không có `1`, không có `-1`, hoặc chúng không bao giờ liền kề thì `best` giữ nguyên `0`, đúng yêu cầu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cặp | `O(n^3)` (xấu nhất) | `O(1)` |
| 2. Mở rộng từ mỗi `1` | `O(n)` | `O(1)` |
| 3. Quét một lần với `last` | `O(n)` | `O(1)` |
| 4. `re` với lookahead | `O(n)` thường gặp | `O(n)` |

`n = len(forts) <= 1000`. Cách 1 chỉ dùng làm oracle với mảng nhỏ trong bộ test.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Mỗi kỳ vọng đều được tính tay (ghi chú bên cạnh).

In [18]:
sols = [SolutionBrute(), SolutionExpand(), SolutionLast(), SolutionRegex(), Solution()]
cases = [
    ([1, 0, 0, -1, 0, 0, 0, 0, 1], 4),     # ví dụ 1
    ([0, 0, 1, -1], 0),                    # ví dụ 2
    ([1], 0),                              # chỉ một phần tử
    ([0], 0),                              # một pháo đài địch
    ([-1], 0),                             # chỉ vị trí trống
    ([1, -1], 0),                          # kề nhau
    ([-1, 1], 0),                          # kề nhau, chiều ngược
    ([1, 0, 0, 0, 0, -1], 4),              # bốn số 0
    ([-1, 0, 0, 1], 2),                    # đi sang trái
    ([1, 0, 0], 0),                        # không có vị trí trống
    ([0, 0, 0], 0),                        # toàn pháo đài địch
    ([1, 0, 0, 1, 0, -1], 1),              # 1 đầu bị chặn bởi 1 thứ hai; 1 thứ hai -> -1 qua một số 0
    ([1, 0, 1, 0, -1], 1),                 # chỉ cặp (2, 4) hợp lệ
    ([-1, 0, 0, 0, 1, 0, -1], 3),          # trái 3, phải 1
    ([-1, -1, 1, 1], 0),                   # cùng loại kề nhau, 1 kề -1
    ([1, 1, 1], 0),                        # không có vị trí trống
]
for forts, want in cases:
    for s in sols:
        got = s.captureForts(forts[:])
        assert got == want, (type(s).__name__, forts, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 16


Phản ví dụ cho các cách làm "ngây thơ": `[1,0,1,0,-1]` có hai số `0` nhưng chỉ một số `0` (cái nằm giữa `1` thứ hai và `-1`) thuộc đường đi hợp lệ; đáp án là `1`, không phải `2`.

In [19]:
assert Solution().captureForts([1, 0, 1, 0, -1]) == 1
assert sum(1 for v in [1, 0, 1, 0, -1] if v == 0) == 2            # số 0 tổng cộng là 2 nhưng đáp án chỉ 1
assert Solution().captureForts([1, 0, 0]) == 0                    # kết thúc không phải -1
print("phản ví dụ OK")

phản ví dụ OK


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `forts` đúng constraints: `1 <= n <= 1000`, giá trị trong `{-1, 0, 1}`. Với mảng nhỏ (`n <= 30`) dùng cả 5 cách (kể cả Brute `O(n^3)`); với mảng lớn (`n` tới 1000) chỉ dùng các cách tuyến tính. Để có nhiều đoạn `0` dài, trọng số của số `0` được tăng.

In [20]:
import random
random.seed(2511)

def gen(n, weights):
    return random.choices([-1, 0, 1], weights=weights, k=n)

for t in range(2500):
    n = random.randint(1, 30)
    w = random.choice([(1, 1, 1), (1, 4, 1), (1, 8, 1), (3, 1, 1)])
    forts = gen(n, w)
    assert 1 <= len(forts) <= 1000 and all(v in (-1, 0, 1) for v in forts)
    ans = [s.captureForts(forts[:]) for s in sols]
    assert len(set(ans)) == 1, (forts, ans)

fast = [SolutionExpand(), SolutionLast(), SolutionRegex(), Solution()]
for t in range(300):
    n = random.randint(500, 1000)
    forts = gen(n, random.choice([(1, 30, 1), (1, 80, 1), (1, 5, 1)]))
    ans = [s.captureForts(forts[:]) for s in fast]
    assert len(set(ans)) == 1, (n, ans)
print("2500 test nhỏ (5 cách) + 300 test lớn (4 cách tuyến tính): khớp nhau")

2500 test nhỏ (5 cách) + 300 test lớn (4 cách tuyến tính): khớp nhau


### 12.3. Kiểm tra giới hạn và cấu trúc đặc biệt

Mảng dài 1000 phần tử đặc biệt: `[1] + [0]*998 + [-1]` có đáp án `998`; `[1] + [0]*999` đáp án `0`; mảng toàn `0` đáp án `0`; đảo mảng không đổi đáp án (vì đề cho phép cả hai chiều).

In [21]:
big = [1] + [0] * 998 + [-1]
assert len(big) == 1000 and Solution().captureForts(big) == 998
assert Solution().captureForts([1] + [0] * 999) == 0
assert Solution().captureForts([0] * 1000) == 0
assert Solution().captureForts([-1] + [0] * 998 + [1]) == 998

random.seed(77)
for _ in range(1000):
    forts = gen(random.randint(1, 40), (1, 3, 1))
    assert Solution().captureForts(forts) == Solution().captureForts(forts[::-1])      # đối xứng theo chiều
    flipped = [-v for v in forts]
    assert Solution().captureForts(forts) == Solution().captureForts(flipped)          # đổi vai 1 và -1
print("các tính chất đối xứng và test biên đều đúng")

các tính chất đối xứng và test biên đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm tất cả các số `0` của mảng

In [22]:
def sai_dem_het_so_0(forts):
    if 1 in forts and -1 in forts:
        return forts.count(0)                           # SAI: đếm mọi pháo đài địch

forts = [1, 0, 0, -1, 0, 0, 0, 0, 1]
print("đúng:", Solution().captureForts(forts), "| sai:", sai_dem_het_so_0(forts))
assert Solution().captureForts(forts) != sai_dem_het_so_0(forts)

đúng: 4 | sai: 6


**Vì sao sai:** chỉ chiếm được các pháo đài **trên đoạn đường đi**, tức một đoạn `0` liên tiếp nằm giữa `1` và `-1`. Mảng có 6 số `0` nhưng đáp án là `4`.

### Lỗi 2: Chỉ xét một chiều (từ trái sang phải, `1` rồi `-1`)

In [23]:
def sai_mot_chieu(forts):
    best, last = 0, -1
    for i, v in enumerate(forts):
        if v == 0:
            continue
        if last != -1 and forts[last] == 1 and v == -1:   # SAI: bỏ qua trường hợp -1 rồi 1
            best = max(best, i - last - 1)
        last = i
    return best

forts = [-1, 0, 0, 1]
print("đúng:", Solution().captureForts(forts), "| sai:", sai_mot_chieu(forts))
assert Solution().captureForts(forts) != sai_mot_chieu(forts)

đúng: 2 | sai: 0


**Vì sao sai:** đề cho phép `j` ở **bên trái** `i`. Quân có thể đi từ `1` sang `-1` ở cả hai hướng, nên cặp `(-1, 1)` liền kề cũng hợp lệ.

### Lỗi 3: Không kiểm tra điểm kết thúc là `-1`

In [24]:
def sai_khong_kiem_tra_dich(forts):
    best, run = 0, 0
    seen_one = False
    for v in forts:
        if v == 1:
            seen_one, run = True, 0
        elif v == 0 and seen_one:
            run += 1
            best = max(best, run)                         # SAI: chưa chắc đoạn 0 kết thúc ở -1
        else:
            seen_one = False
    return best

forts = [1, 0, 0]
print("đúng:", Solution().captureForts(forts), "| sai:", sai_khong_kiem_tra_dich(forts))
assert Solution().captureForts(forts) != sai_khong_kiem_tra_dich(forts)

đúng: 0 | sai: 2


**Vì sao sai:** đoạn `0` sau một số `1` chỉ có ích nếu **kết thúc** ở một vị trí **trống** `-1`. Nếu mảng kết thúc hoặc gặp `1` khác thì không có vị trí đích.

### Lỗi 4: Lấy khoảng cách giữa số `1` đầu và số `-1` cuối

In [25]:
def sai_dau_cuoi(forts):
    if 1 not in forts or -1 not in forts:
        return 0
    i = forts.index(1)
    j = len(forts) - 1 - forts[::-1].index(-1)
    return max(abs(j - i) - 1, 0)                        # SAI: bỏ qua các phần tử khác 0 nằm giữa

forts = [1, 0, 1, 0, -1]
print("đúng:", Solution().captureForts(forts), "| sai:", sai_dau_cuoi(forts))
assert Solution().captureForts(forts) != sai_dau_cuoi(forts)

đúng: 1 | sai: 3


**Vì sao sai:** giữa `1` đầu và `-1` cuối còn có một số `1` khác chặn đường. Đường đi hợp lệ chỉ đi qua toàn số `0`.

### Lỗi 5: Nối hai phần tử cùng loại

In [26]:
def sai_cung_loai(forts):
    best, last = 0, -1
    for i, v in enumerate(forts):
        if v == 0:
            continue
        if last != -1:
            best = max(best, i - last - 1)               # SAI: không kiểm tra khác dấu
        last = i
    return best

forts = [1, 0, 0, 1]
print("đúng:", Solution().captureForts(forts), "| sai:", sai_cung_loai(forts))
assert Solution().captureForts(forts) != sai_cung_loai(forts)

đúng: 0 | sai: 2


**Vì sao sai:** `1` với `1` (hoặc `-1` với `-1`) không phải đường đi từ pháo đài của ta đến vị trí trống. Phải có `forts[last] != forts[i]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "chỉ đi qua các số `0` giữa hai điểm đặc biệt" ⇒ **hai phần tử khác 0 liền kề** cùng một lượt quét, dùng biến `last`.
- Điều kiện "khác dấu" `forts[last] != v` gộp cả hai chiều đi.
- Đáp án mặc định `0`: không cần xử lý riêng "không có `1`" hoặc "không có `-1`".
- Quét theo nhóm: `itertools.groupby` cũng giải được (gom các run các số `0`).
- Khi `n <= 1000` vẫn nên tránh `O(n^3)`; viết cách tuyến tính ngay.
- Test nhanh với `[1,0,0,-1,0,0,0,0,1] -> 4` và `[0,0,1,-1] -> 0`.

## 15. Biến thể của bài

1. **Trả về cặp vị trí** `(i, j)` thực hiện chiếm lớn nhất, thay vì chỉ số lượng.
2. **Đếm số cách di chuyển** đạt giá trị lớn nhất.
3. **Cho phép đi qua cả pháo đài của ta** (giá trị `1` ở giữa): khi đó chặn chỉ do phần tử `-1`.
4. **Chiếm nhiều lần liên tiếp:** sau khi tới `j`, `j` trở thành pháo đài của ta, tìm dãy lần chiếm có tổng lớn nhất.
5. **Phiên bản 2D:** lưới với đường đi chỉ qua ô địch (BFS), tìm số ô địch tối đa chiếm được.
6. **Dạng trực tuyến:** các phần tử đến lần lượt; chỉ cần giữ `last` và `lastVal`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | kiểm đoạn giữa toàn 0 | `O(n^3)` | `O(1)` | oracle cho mảng nhỏ |
| 2. Mở rộng từ mỗi `1` | đi hai phía qua các `0` | `O(n)` | `O(1)` | khi muốn bám sát mô tả đề |
| 3. Quét một lần `last` | cặp khác 0 liền kề, khác dấu | `O(n)` | `O(1)` | cách nộp bài chuẩn |
| 4. `re` lookahead | mẫu `A0*B` và `B0*A` | `O(n)` | `O(n)` | khi quen với chuỗi và regex |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().captureForts([1, 0, 0, 0, -1])` bằng bao nhiêu? Còn `[-1, 0, 1, 0, 0, 0, -1]`?

**Bài 2 (tính tay).** Với `forts = [0, 1, 0, 0, -1, 0, 1, -1]`, liệt kê các cặp phần tử khác 0 liền kề, loại cặp không hợp lệ và tìm đáp án.

**Bài 3 (điền TODO).** Hoàn thành `best_pair(forts)` trả về `(i, j)` (chỉ số của hai phần tử khác 0 tạo ra lần chiếm lớn nhất, `i < j`), hoặc `None` nếu đáp án là `0`.

In [27]:
def best_pair(forts):
    best, pair, last = 0, None, -1
    # TODO: quét như Cách 3; khi tìm được cặp khác dấu có i - last - 1 > best thì cập nhật pair = (last, i)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_valid_moves(forts)` đếm số cặp phần tử khác 0 liền kề **khác dấu** (số lần di chuyển hợp lệ, kể cả lần chiếm `0` pháo đài).

**Bài 5 (biến thể / mini challenge).** Viết `capture_allow_own(forts)`: giống đề nhưng quân được đi qua cả pháo đài của ta (`1`), chỉ bị chặn bởi vị trí trống `-1` ở giữa. Đi từ một `1` đến một `-1` sao cho giữa **không có** `-1`; số pháo đài chiếm là số `0` trên đoạn. Kiểm tra `capture_allow_own([1, 0, 1, 0, -1]) == 2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1,0,0,0,-1]` có ba số 0 nên `3`; `[-1,0,1,0,0,0,-1]` có cặp `(-1,1)` chiếm 1 và `(1,-1)` chiếm 3 nên `3`.
- **Mức 2 (vừa):** Bài 2: phần tử khác 0: `(1:1) (4:-1) (6:1) (7:-1)`; cặp `1->-1` chiếm `2`, `-1->1` chiếm `1`, `1->-1` chiếm `0`; đáp án `2`. Bài 3: cập nhật `pair` cùng lúc cập nhật `best`.
- **Mức 3 (sâu):** Bài 4: đếm số `forts[last] != v`. Bài 5: với mỗi `1`, đi sang hai phía qua các số `0` và `1`; dừng khi gặp `-1` (hợp lệ) hoặc hết mảng; đếm số `0` trên đường đi, không đếm các số `1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert Solution().captureForts([1, 0, 0, 0, -1]) == 3
assert Solution().captureForts([-1, 0, 1, 0, 0, 0, -1]) == 3

# Bài 2
assert Solution().captureForts([0, 1, 0, 0, -1, 0, 1, -1]) == 2

# Bài 3
def best_pair(forts):
    best, pair, last = 0, None, -1
    for i, v in enumerate(forts):
        if v == 0:
            continue
        if last != -1 and forts[last] != v and i - last - 1 > best:
            best = i - last - 1
            pair = (last, i)
        last = i
    return pair

# Bài 4
def count_valid_moves(forts):
    cnt, last = 0, -1
    for i, v in enumerate(forts):
        if v == 0:
            continue
        if last != -1 and forts[last] != v:
            cnt += 1
        last = i
    return cnt

# Bài 5
def capture_allow_own(forts):
    n = len(forts)
    best = 0
    for i in range(n):
        if forts[i] != 1:
            continue
        for step in (1, -1):
            j, zeros = i + step, 0
            while 0 <= j < n and forts[j] != -1:
                if forts[j] == 0:
                    zeros += 1
                j += step
            if 0 <= j < n:                                 # dừng ở -1
                best = max(best, zeros)
    return best

assert best_pair([1, 0, 0, -1, 0, 0, 0, 0, 1]) == (3, 8)
assert best_pair([0, 0, 1, -1]) is None
assert best_pair([-1, 0, 0, 1]) == (0, 3)
assert count_valid_moves([1, 0, 0, -1, 0, 0, 0, 0, 1]) == 2
assert count_valid_moves([1, 1, -1, -1]) == 1
assert count_valid_moves([0, 0]) == 0
assert capture_allow_own([1, 0, 1, 0, -1]) == 2
assert capture_allow_own([1, 0, 0, -1]) == 2
assert capture_allow_own([1, 0, 0]) == 0
random.seed(25)
for _ in range(300):
    f = gen(random.randint(1, 20), (1, 3, 1))
    pr = best_pair(f)
    if pr is None:
        assert Solution().captureForts(f) == 0
    else:
        assert pr[1] - pr[0] - 1 == Solution().captureForts(f)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quét một lần, nhớ phần tử khác 0 gần nhất)

```python
class Solution:
    def captureForts(self, forts: List[int]) -> int:
        best, last = 0, -1
        for i, v in enumerate(forts):
            if v == 0:
                continue
            if last != -1 and forts[last] != v:
                best = max(best, i - last - 1)
            last = i
        return best
```

**Checklist**

- [ ] Chỉ xét cặp phần tử khác 0 **liền kề** (không có số khác 0 ở giữa)?
- [ ] Điều kiện cặp: **khác dấu** (`1` với `-1`), cả hai chiều?
- [ ] Giá trị chiếm được là `i - last - 1` (số `0` ở giữa), không phải `i - last`?
- [ ] Đáp án mặc định `0` khi không có cặp hợp lệ?
- [ ] Không đếm toàn bộ số `0` của mảng?
- [ ] Đã thử `[1,0,0,-1,0,0,0,0,1] -> 4` và `[0,0,1,-1] -> 0`?
- [ ] Signature đúng `captureForts(self, forts: List[int]) -> int` và có `from typing import List`?